# 1. Apriori
### Limpieza de los datos
Para la implementacion de nuestro algoritmo de Apriori, quitamos los datos que aparecian con la etiqueta de "CONFIDENCIAL" y como Apriori no trabaja con numeros continuos calculamos la EDAD y discretamos la EDAD para poder trabajar en categorias









In [27]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("data_secretariado.csv")

# Limpiar datos
df = df.replace("CONFIDENCIAL", pd.NA).dropna()

# Convertir fechas
df['FECHA_NAC_DT'] = pd.to_datetime(df['FECHA_NACIMIENTO'], errors='coerce')
df['FECHA_DESAP_DT'] = pd.to_datetime(df['FECHA_DESAPARICION'], errors='coerce')

# Calcular edad
df['EDAD_CALCULADA'] = df['FECHA_DESAP_DT'].dt.year - df['FECHA_NAC_DT'].dt.year

bins_edad = [0, 11, 17, 29, 59, 120]
labels_edad = ['0-11', '12-17', '18-29', '30-59', '60+']

df['GRUPO_EDAD'] = pd.cut(df['EDAD_CALCULADA'], bins=bins_edad, labels=labels_edad)

df['MES_NUMERO'] = df['FECHA_DESAP_DT'].dt.month

bins_mes = [0, 3, 6, 9, 12]
labels_mes = ['T1_Ene-Mar', 'T2_Abr-Jun', 'T3_Jul-Sep', 'T4_Oct-Dic']

df['PERIODO_DESAPARICION'] = pd.cut(df['MES_NUMERO'], bins=bins_mes, labels=labels_mes)

df = df[['SEXO', 'ENTIDAD', 'MUNICIPIO', 'ESTATUS_VICTIMA', 'GRUPO_EDAD', 'PERIODO_DESAPARICION']].dropna()

transactions = df.values.tolist()



In [24]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df_encoded = pd.DataFrame(te_array, columns=te.columns_)

frequent_itemsets = apriori(df_encoded, min_support=0.1, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)

print(rules[['antecedents', 'consequents', 'support', 'confidence']])

                          antecedents             consequents   support  \
0                             (18-29)          (DESAPARECIDA)  0.333037   
1                             (18-29)                (HOMBRE)  0.282555   
2                             (30-59)          (DESAPARECIDA)  0.445967   
3                             (30-59)                (HOMBRE)  0.409790   
4                  (ESTADO DE MÉXICO)          (DESAPARECIDA)  0.142085   
5                            (HOMBRE)          (DESAPARECIDA)  0.745511   
6                      (DESAPARECIDA)                (HOMBRE)  0.745511   
7                             (MUJER)          (DESAPARECIDA)  0.198358   
8                        (T1_Ene-Mar)          (DESAPARECIDA)  0.223558   
9                        (T2_Abr-Jun)          (DESAPARECIDA)  0.251523   
10                       (T3_Jul-Sep)          (DESAPARECIDA)  0.267544   
11                       (T4_Oct-Dic)          (DESAPARECIDA)  0.202272   
12                       

In [28]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df_encoded = pd.DataFrame(te_array, columns=te.columns_)

frequent_itemsets = apriori(df_encoded, min_support=0.1, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)

rules_filtradas = rules[(rules['confidence'] >= 0.6) & (rules['lift'] > 1)].sort_values(by='lift', ascending=False)

In [31]:
def format_rules(df):
    tabla = df.copy()

    # Convertir sets a texto
    tabla['antecedents'] = tabla['antecedents'].apply(lambda x: ', '.join(list(x)))
    tabla['consequents'] = tabla['consequents'].apply(lambda x: ', '.join(list(x)))

    # Redondear métricas
    tabla['support'] = tabla['support'].round(3)
    tabla['confidence'] = tabla['confidence'].round(3)
    tabla['lift'] = tabla['lift'].round(3)

    return tabla[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

tabla_final = format_rules(rules_filtradas)

print(tabla_final.head(20))

                        antecedents           consequents  support  \
41                30-59, T2_Abr-Jun  HOMBRE, DESAPARECIDA    0.103   
25                30-59, T2_Abr-Jun                HOMBRE    0.109   
40  30-59, DESAPARECIDA, T2_Abr-Jun                HOMBRE    0.103   
3                             30-59                HOMBRE    0.410   
21                            30-59  HOMBRE, DESAPARECIDA    0.385   
20              30-59, DESAPARECIDA                HOMBRE    0.385   
26                30-59, T3_Jul-Sep                HOMBRE    0.114   
43  30-59, DESAPARECIDA, T3_Jul-Sep                HOMBRE    0.107   
44                30-59, T3_Jul-Sep  HOMBRE, DESAPARECIDA    0.107   
4                  ESTADO DE MÉXICO          DESAPARECIDA    0.142   
32                       T2_Abr-Jun  HOMBRE, DESAPARECIDA    0.200   
7                             MUJER          DESAPARECIDA    0.198   
1                             18-29                HOMBRE    0.283   
18                  